## 2. Demand Estimation

We will estimate a logit-style demand model using linear regression. The model is:

$$
\log(s_{bt}) = \alpha_0 + \alpha_t + \gamma_b + \beta_{price}p_{bt} + \beta_{rating}r_{bt} + \sum_{\ell=1}^L \beta_\ell x_{bt\ell} + \epsilon_{bt}.
$$

Here:

- $b$ indexes brands
- $t$ indexes years
- $s_{bt}$ is `brand_share`
- $p_{bt}$ is `avg_price`
- $r_{bt}$ is `avg_rating`
- $x_{bt\ell}$ are the product characteristics
- $\alpha_t$ are year dummy coefficients
- $\gamma_b$ are brand dummy coefficients
- $\beta_{price}$ is **one constant price coefficient**, shared by all brands and all years

That last point matters: do **not** estimate a different price coefficient for every brand-year. We do not have enough information for that, and it would make the cost calculation impossible to interpret.

Use `pd.get_dummies(..., drop_first=True)` for brand and year dummies. The dropped brand and dropped year become the reference categories, so all dummy coefficients are interpreted relative to those omitted categories.

Questions:

1. What is the estimated price coefficient, $\hat{\beta}_{price}$?
2. Is it negative? Why is that important?
3. Which product features are associated with higher demand?
4. Which brand dummy coefficients are largest? Remember that these are interpreted relative to the dropped brand.
5. Which year dummy coefficients are largest? Remember that these are interpreted relative to the dropped year.
6. What is the model's $R^2$?

This part of the work is the **data scientist** role: turning the cleaned data into a model that can be used for prediction and interpretation.

In [11]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

df = pd.read_csv('air_fryers_clean_brand_year.csv')
df.head()

,category,year,brand,purchase_count,product_count,avg_price,avg_rating,compact_share,dual_basket_share,oven_style_share,rotisserie_share,window_share,market_purchases,brand_share,log_brand_share
0,air_fryers,2019,chefman,1146,10,72.963695,4.434119,1.000000,0.0,0.780977,0.243455,0.184119,15076,0.076015,-2.576826
1,air_fryers,2019,cosori,11,2,159.990000,4.581818,1.000000,0.0,0.090909,0.090909,0.000000,15076,0.000730,-7.222964
2,air_fryers,2019,cuisinart,1616,22,229.465274,4.481312,0.993812,0.0,0.889851,0.000000,0.000000,15076,0.107190,-2.233150
3,air_fryers,2019,dash,3011,19,55.176333,4.390767,1.000000,0.0,0.973431,0.000000,0.000000,15076,0.199721,-1.610832
4,air_fryers,2019,gowise usa,4405,45,83.575551,4.552259,0.999773,0.0,0.129398,0.128490,0.000000,15076,0.292186,-1.230364


In [12]:
feature_cols = [
    "avg_price",
    "avg_rating",
    "brand_share",]

In [13]:
y = df['log_brand_share']
brand_dummies = pd.get_dummies(df['brand'],
                               prefix='brand', drop_first=True, dtype=int)
year_dummies = pd.get_dummies(df['year'].astype(str),
                              prefix='year', drop_first=True, dtype=int)

X = pd.concat(
    [df[['avg_price', 'avg_rating'] + feature_cols],
     brand_dummies,
     year_dummies],
    axis=1,
)

model=LinearRegression()
model.fit(X, y)

predicted_log_share = model.predict(X)
r2 = r2_score(y, predicted_log_share)

coef_table = pd.DataFrame({
  'feature': X.columns,
  'coefficient': model.coef_})

print('R-squared:', r2)
coef_table

R-squared: 0.8868976586329154


,feature,coefficient
0,avg_price,-0.011339
1,avg_rating,1.167429
2,avg_price,-0.011339
3,avg_rating,1.167429
4,brand_share,10.642977
5,brand_cosori,-1.113140
6,brand_cuisinart,3.009323
7,brand_dash,-0.955881
8,brand_gowise usa,-0.668241
9,brand_instant_pot,-0.533281


Part 1: <br>
The estimated price coefficient is -0.011339.

Part 2: <br>
Yes, the price coefficient is negative, which shows that as price increases, demand decreases while everything else is constant. This means that with every 1 unit increase in price, demand decreases by 0.0113 units.

Part 3: <br>
avg_rating and brand_share are associated with higher demand. avg_rating has a coefficient of 1.167429, which shows a positive effect. brand_share has a coefficient of 10.642977, which shows a very strong positive effect.This makes sense since higher review ratings and larger market share would typically be associated with more people buying the product.

Part 4: <br>
The brand dummy coefficients that are the largest are Cuisinart and Oster. This means that they are the most strongly correlated with demand, compared to the baseline brand.

Part 5: <br>
The year dummy coefficients that are the largest are 2021 and 2022. This means that they are the most strongly correlated with demand, compared to the baseline year.

Part 6: <br>
The model's R^2 is 0.887. This means that 88.7% of the variation in log_brand_share can be explained by the model.